In [1]:
import opendatasets as od

# This triggers a Kaggle login prompt (username + API key) the first time.
# On Windows, if prompted for a "kaggle.json" file location, it typically expects
# it at C:\Users\<YourUsername>\.kaggle\kaggle.json — download this API key file
# from your Kaggle account settings page ("Create New API Token") beforehand.
dataset_url = 'https://www.kaggle.com/c/new-york-city-taxi-fare-prediction/overview'
od.download(dataset_url)

data_dir = './new-york-city-taxi-fare-prediction'

Skipping, found downloaded files in ".\new-york-city-taxi-fare-prediction" (use force=True to force download)


In [ ]:
import pandas as pd

# Only read the columns we actually need, with the SMALLEST dtype that safely fits each column.
# float32 instead of the pandas default float64 halves memory for coordinate/fare columns.
# uint8 for passenger_count since it never exceeds ~9.
dtypes = {
    'fare_amount': 'float32',
    'pickup_longitude': 'float32',
    'pickup_latitude': 'float32',
    'dropoff_longitude': 'float32',
    'dropoff_latitude': 'float32',
    'passenger_count': 'uint8'
}

# nrows caps how many rows pandas even parses from disk — this alone is the biggest speed win.
# parse_dates converts pickup_datetime straight into a proper datetime64 column instead of a string.
df = pd.read_csv(
    f'{data_dir}/train.csv',
    nrows=5_000_000,
    dtype=dtypes,
    parse_dates=['pickup_datetime']
)

# Case study explicitly suggests working with ~1% of data for iteration speed.
# random_state=42 is set for reproducibility — same sample every run, so results are comparable across experiments.
df = df.sample(frac=0.10, random_state=42)  # 10% of the 500k pre-filtered rows ≈ 50k rows, a workable size
print(df.shape)

In [ ]:
df.info()          # column dtypes, non-null counts — quick sanity check
df.describe()       # min/max/mean per numeric column — this is where garbage jumps out
df.isnull().sum()   # exact count of missing values per column

In [ ]:
# Concrete cleaning step based on the audit above
df = df[
    (df.fare_amount > 0) &
    (df.passenger_count > 0) & (df.passenger_count <= 6) &
    (df.pickup_latitude.between(40.5, 40.9)) &
    (df.dropoff_latitude.between(40.5, 40.9)) &
    (df.pickup_longitude.between(-74.3, -73.7)) &
    (df.dropoff_longitude.between(-74.3, -73.7))
].copy()

In [ ]:
# .dt accessor extracts date/time components from a datetime64 column
df['hour'] = df.pickup_datetime.dt.hour                 # 0–23
df['day_of_week'] = df.pickup_datetime.dt.day_name()     # 'Monday', 'Tuesday', ...
df['day_of_week_num'] = df.pickup_datetime.dt.dayofweek  # 0=Monday ... 6=Sunday (numeric, needed for modeling later)
df['month'] = df.pickup_datetime.dt.month                # 1–12

In [ ]:
import matplotlib.pyplot as plt

ride_counts_by_day = df.day_of_week.value_counts()
ride_counts_by_day.plot(kind='bar', title='Number of Rides by Day of Week')
plt.ylabel('Ride Count')
plt.show()

In [ ]:
df.hour.value_counts().sort_index().plot(kind='line', title='Ride Volume by Hour of Day')
plt.xlabel('Hour (24h)')
plt.ylabel('Ride Count')
plt.show()

In [ ]:
df.groupby('month').fare_amount.mean().plot(kind='bar', title='Average Fare by Month')
plt.ylabel('Average Fare ($)')
plt.show()

In [ ]:
# Round coordinates to ~1km precision to group nearby drop points into buckets
df['dropoff_lat_rounded'] = df.dropoff_latitude.round(2)
df['dropoff_lon_rounded'] = df.dropoff_longitude.round(2)

top_dropoffs = (
    df.groupby(['dropoff_lat_rounded', 'dropoff_lon_rounded'])
      .fare_amount.mean()
      .sort_values(ascending=False)
      .head(10)
)
print(top_dropoffs)
# Cross-reference the highest lat/long pairs against a map (e.g., Google Maps) —
# expect to see JFK/LaGuardia airport coordinates surface here, since airport rides are flat-rate and long-distance.

In [ ]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    """
    Computes great-circle distance (in km) between two lat/long points on Earth's surface.
    This matters because Euclidean distance on raw lat/long degrees is WRONG —
    degrees of longitude shrink in real-world distance as you move away from the equator,
    so a naive sqrt((lat2-lat1)^2 + (lon2-lon1)^2) would misrepresent true distance.
    The haversine formula accounts for the Earth's curvature.
    """
    R = 6371  # Earth's radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])  # trig functions need radians, not degrees
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df['distance_km'] = haversine(
    df.pickup_latitude, df.pickup_longitude,
    df.dropoff_latitude, df.dropoff_longitude
)

# A distance of exactly 0 means pickup == dropoff, which is not a valid fare-generating ride — drop these.
df = df[df.distance_km > 0]

# Cap unrealistic distances (data errors, not real NYC trips)
df = df[df.distance_km < 100]

In [ ]:
from sklearn.model_selection import train_test_split

# These are the exact 5 features referenced in the case study's "Model Explanation" section,
# transformed into model-friendly numeric form.
features = ['distance_km', 'hour', 'day_of_week_num', 'month', 'passenger_count']
X = df[features]
y = df['fare_amount']

# test_size=0.2 -> 80% train / 20% test, a standard default split.
# random_state=42 -> reproducible split; without it, every run would shuffle differently
# and RMSE comparisons between models would not be a fair comparison.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
from sklearn.metrics import root_mean_squared_error

# The "dumbest possible" model: ignore all inputs, always guess the average fare.
# This sets the floor — any real model MUST beat this to be worth using at all.
mean_fare = y_train.mean()
baseline_predictions = [mean_fare] * len(y_test)

baseline_rmse = root_mean_squared_error(y_test, baseline_predictions)
print(f"Hardcoded baseline RMSE: ${baseline_rmse:.2f}")

In [ ]:
from sklearn.linear_model import LinearRegression

# Linear Regression assumes fare = w1*distance + w2*hour + w3*day + w4*month + w5*passengers + b
# It's fast, interpretable (each weight tells you how much that feature moves the fare),
# and a natural "first real model" to compare against the hardcoded baseline.
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)
lr_rmse = root_mean_squared_error(y_test, lr_predictions)
print(f"Linear Regression RMSE: ${lr_rmse:.2f}")

# Inspect learned weights — useful talking point: "distance_km should have by far the largest coefficient"
for feature, coef in zip(features, lr_model.coef_):
    print(f"{feature}: {coef:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest = an ensemble of many decision trees, each trained on a random subset of rows/features,
# with predictions averaged across all trees. This captures NON-linear relationships
# (e.g., fare might jump sharply for very long distances in a way a straight line can't represent)
# and is generally more accurate than plain linear regression on messy real-world data.
rf_model = RandomForestRegressor(
    n_estimators=500,   # number of trees — more trees = more stable predictions, at the cost of training time
    max_depth=10,       # caps how deep each tree can grow — controls overfitting (see note below)
    random_state=42,
    n_jobs=-1           # use all CPU cores to speed up training
)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_rmse = root_mean_squared_error(y_test, rf_predictions)
print(f"Random Forest RMSE: ${rf_rmse:.2f}")

# Feature importance — which inputs actually drove the predictions the most
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

In [ ]:
results = pd.DataFrame({
    'Model': ['Hardcoded (mean)', 'Linear Regression', 'Random Forest'],
    'RMSE': [baseline_rmse, lr_rmse, rf_rmse]
})
print(results)

In [ ]:
import joblib
joblib.dump(rf_model, 'fare_model.pkl')